## Linear Regression



While Excel and many Python plotting libraries offer straightforward methods to plot a regression line, conducting a thorough analysis of linear regression is actually quite complex. In this module, we will demonstrate the proper approach to linear regression analysis. Specifically, we will examine whether the data and model residuals satisfy the assumptions of linear regression, how to visualize confidence intervals for the regression model, and, as a valuable addition, how to plot the prediction intervals for new observations derived from the linear regression model.



### Causation versus Correlation



Back in my home country, and before the hippy movement changed our culture, kids, who were curious about where the babies come from, were told that they are brought by the stork (a large bird, see Fig.ref:fig:storksa). Storks were indeed a common sight in rural areas, and large enough to sell this story to a 3-year-old.

![img](./Ringed_white_stork.png)

As mature scientists who value critical thinking, we seek data to support this idea. Specifically, we expect to find a strong correlation between the number of storks and the number of babies born. Surprisingly, these two variables do indeed correlate in a statistically significant manner. Countries with larger stork populations tend to have higher birthrates. This phenomenon, where both variables increase together, is referred to as a positive correlation. See Fig. [BROKEN LINK: fig:storks]

![img](./figures/stork-new.png)

This does not prove that storks deliver babies. Clearly, the correlation between two observable quantities does not imply causation. A more plausible explanation is that both variables are influenced by a shared factor, such as industrialization. It is a common mistake to confuse correlation with causation. Another good example is to correlate drinking with heart attacks. This surely will correlate, but the story is more complex. Are there, e.g., patterns like drinkers tending to do less exercise than non-drinkers? So even if you have a good hypothesis for why two variables are correlated, the correlation itself proves nothing.

 
Regardless of a causal relationship, the linear correlation between two datasets can be quantified using the Pearson Correlation Coefficient (PCC, commonly referred to as `r`). This coefficient indicates both the strength and direction of a **linear** relationship between two variables. The PCC ranges from +1 (indicating a perfect positive linear correlation) to -1 (indicating a perfect negative or inverse linear correlation). Correlations are often considered weak when they fall between +0.3 and -0.3, and strong when they exceed +0.7 or are lower than -0.7 (though thresholds depend on the field). It is important to note that Pearson&rsquo;s `r` specifically measures linear association: two variables can have a strong non-linear relationship (e.g., quadratic) and still have r close to 0. (To assess general monotonic relationships without assuming linearity, rank-based measures such as Spearman&rsquo;s &rho; or Kendall&rsquo;s &tau; are used instead.)
We can use the `corr()` method of a pandas Series to calculate the PCC:



In [1]:
import pandas as pd

df = pd.read_csv("storks_vs_birth_rate.csv").set_axis(["Babies", "Storks"], axis=1)
r = df["Storks"].corr(df["Babies"])
print(f"r = {r:.2f}")

Based on this analysis, we can confirm that the number of babies and storks correlate.



### Understanding the results of a linear regression analysis



Linear regression analysis builds upon correlation analysis by evaluating how effectively a linear function, such as a straight line, represents the relationship between two variables, `x` and `y`. A straight line is described by the equation `y = mx + b`, where adjustments are made to the coefficients `m` (slope) and `b` (y-intercept) to minimize the difference between the observed data points and the predictions made by the model. Numerous libraries are available to facilitate this fitting process, and ideally, they provide not only the resulting linear model (the equation of the fitted line) but also additional parameters that assess the quality of the fit.

The most common parameter is the coefficient of determination, denoted as r<sup>2</sup> (or R<sup>2</sup>). This metric indicates how effectively a linear relationship explains the variability in the dependent variable `y` using a single independent variable `x`. In the figure above, r<sup>2</sup> = 0.38, suggesting that 38% of the variation in the number of newborn babies can be accounted for by a linear relationship with the number of storks. Often, however, multiple independent variables are needed to better capture the variability in the data. In such cases, multiple linear regression is utilized, represented as `y = b + m1*x1 + m2*x2 + ... + mn*xn + \varepsilon`. This approach involves solving a system of linear equations (a concept from linear algebra) and provides the multiple coefficient of determination, denoted as $R^2$, which quantifies the proportion of variability in `y` explained by the combination of independent variables $x_i$. Many statistical programs use capital $R_2$ even for simple (single-variable) linear regression, which is standard practice; lowercase $r^2$is sometimes used in the single-variable case to highlight that it equals the square of Pearson&rsquo;s $r$.

From a user perspective, we are interested in understanding how good the model is and how to interpret the key indicators of a given regression model:

-   **r<sup>2</sup> (or R<sup>2</sup>):** the coefficient of determination. This value lies between 0 and 1 and expresses the proportion of observed variance in `y` explained by the regression model. A value of r<sup>2</sup> = 0.7 indicates that 70% of the variance in `y` is explained by the linear model, while 30% is attributable to other factors not captured by the model (e.g., measurement error, omitted variables, or non-linear effects). In Fig. [BROKEN LINK: fig:storks], 38% of the variance in birthrate is accounted for by the linear relationship with stork pairs.
-   **p:** In simple linear regression, we test the **null hypothesis** (H<sub>0</sub>) that the true slope is zero (i.e., `y` has no linear relationship with `x`). The p-value is the probability of observing a slope at least as extreme as the one in our sample, **assuming the null hypothesis is true**. Importantly, the p-value is **not** the probability that the null hypothesis is true. A small p-value (commonly `p < 0.05`, corresponding to a significance level of &alpha; = 0.05) indicates that the observed data would be unlikely under H<sub>0</sub>, leading us to reject H<sub>0</sub> in favor of a statistically significant linear relationship. In Fig. [BROKEN LINK: fig:storks], `p = 0.008` which are well below 0.05. It is important to note that r<sup>2</sup> (effect size / explained variance) and p (statistical significance) measure different things: a small r<sup>2</sup> can still have a small p-value if the sample size is large.



### Testing the data



#### Histograms




The Storks dataset is strongly right-skewed rather than symmetric like a [Gaussian Normal Distribution](https://en.wikipedia.org/wiki/Normal_distribution) (bell curve). Note that ordinary least squares (OLS) linear regression does **not** require the raw x or y variables themselves to be normally distributed—the normality assumption applies to the **residuals** (discussed below). However, inspecting [histograms](https://en.wikipedia.org/wiki/Histogram) of x and y is still an essential exploratory step: strongly skewed variables or extreme values can create high-leverage points that disproportionately influence the fitted line, in which case a log-transformation may be appropriate.



In [1]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv("storks_vs_birth_rate.csv").set_axis(["Babies", "Storks"], axis=1)

fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, layout="constrained")
ax1.hist(df["Babies"])
ax1.set_title("Babies")
ax2.hist(df["Storks"])
ax2.set_title("Storks")
fig.savefig("strk_histogram.png")
plt.show()

![img](./figures/strk_histogram.png)

As shown in the histograms, both variables are strongly right-skewed with a few large values. We will keep this in mind as we fit our model and inspect the residuals.



### The statsmodels library



Python&rsquo;s success largely stems from its vast array of third-party libraries.  If all you need is a quick regression line plot, use the  [seaborn statistical data visualization](https://seaborn.pydata.org/) library. If you need a linear regression for only two variables, as well as basic model parameters like r<sup>2</sup>, use [scipy](https://docs.scipy.org/) which also allows for a straightforward way to do non-linear curve fitting. However, for this exercise we will bring in the heavy guns and use the [statsmodel libabry.](https://www.statsmodels.org/) I do this because:

-   Quick and dirty linear regression analysis is readily available in Excel and the likes, but the stats model library allows us to go beyond and include some rather useful output.
-   He statsmodels library provides an R-style interface that allows for more complex regression analysis of multiple variables.
-   Regression analysis examples with seaborn, scipy or [scikit-learn](https://scikit-learn.org/) are readily available on the internet, but statsmodel examples are harder to find.
-   Alas because of its size and complexity, the statsmodels library does not support type hinting.

The statsmodels library offers several interfaces, and we will focus on the formula interface, which resembles R&rsquo;s formula syntax (for more details, see [https://www.statsmodels.org/dev/example_formulas.html](https://www.statsmodels.org/dev/example_formulas.html)). However, it&rsquo;s important to note that not all functions in statsmodels are accessible through this interface at this time.

For the regression model, we aim to investigate whether the number of storks correlates with the number of babies born. In other words, does the birth rate depend on the stork population? To achieve this, we must define a statistical model and evaluate how well its predictions align with the data. 

To make our code less dependent on the column names in whatever spreadsheet you are using, we will replace the specific data-dependent names with general aliases: `X` for the independent variable (&ldquo;Storks&rdquo;) and `Y` for the dependent variable (&ldquo;Babies&rdquo;). After importing the data, we specify the model using the formula `"Y ~ X"` (meaning `Y` is modeled as a linear function of `X`) and immediately call `.fit()` to fit the OLS model. Calling `.summary()` prints a comprehensive table of regression diagnostics.



In [1]:
import pandas as pd
import statsmodels.formula.api as smf

df = pd.read_csv("storks_vs_birth_rate.csv").set_axis(["Y", "X"], axis=1)
results = smf.ols("Y ~ X", data=df).fit()
print(results.summary())

Plenty of information here, probably more than you asked for. Let&rsquo;s tease out the important ones:

-   The first line states that `Y` is the dependent variable. This is useful and will help you catch errors in your model definition.
-   The second line confirms that this is an ordinary least squares (OLS) model.
-   At the bottom, there are diagnostic warnings (for example, regarding sample size or skewness/kurtosis of the residuals) that flag potential issues with model assumptions.

If you compare the output with Figure [ref:fig:storks](ref:fig:storks), you can see that r<sup>2</sup> is called &ldquo;R-squared&rdquo;, the p-value is called &ldquo;Prob (F-statistic)&rdquo; (which equals the t-test p-value for the slope `X` in simple linear regression), the y-intercept is in the &ldquo;Intercept&rdquo; row, and the slope is in the &ldquo;X&rdquo; row. You can extract these parameters from the results object like this:



In [1]:
slope = results.params["X"]
y_0 = results.params["Intercept"]
r_square = results.rsquared
p_value = results.pvalues["X"]

### Testing the validity of your analysis



Just because you can fit a regression line does not imply that a linear model is appropriate. Key assumptions of OLS linear regression include linearity, constant variance of errors (homoscedasticity), independence, and—for exact small-sample inference (p-values and intervals)—normality of the **residuals**. Residuals are defined as observed minus predicted values ($e_i = y_i - \hat{y}_i$), and `statsmodels` provides them directly via `results.resid`. Plotting residuals against fitted values should ideally show random scatter around 0 with no funnel shape or curvature.



In [1]:
fig, ax = plt.subplots(layout="constrained")
ax.scatter(results.fittedvalues, results.resid)
ax.axhline(0, color="C1", linestyle="--")
ax.set(xlabel="Fitted Values", ylabel="Residuals", title="Residuals vs. Fitted")
plt.show()

#### QQplots




Another visual method to assess whether the residuals follow a normal distribution is a [Q-Q (quantile-quantile) plot](https://en.wikipedia.org/wiki/Q%E2%80%93Q_plot). 



In [1]:
import statsmodels.api as sm

fig, ax = plt.subplots(layout="constrained")
sm.qqplot(results.resid, ax=ax, line="s")
fig.savefig("qqplot.png")
plt.show()

Here&rsquo;s how to interpret a Q-Q plot:

1.  **Axes**:
    -   The x-axis represents the theoretical quantiles from a standard normal distribution.
    -   The y-axis represents the observed quantiles of the residuals.

2.  **Points & Reference Line**:
    -   Each point pairs a sample quantile with its corresponding theoretical quantile.
    -   If the residuals are normally distributed, the points will lie closely along the straight reference line.

3.  **Interpreting Patterns**:
    -   **Straight Line**: Residuals are consistent with a normal distribution.
    -   **S-Curve**: Points curving away in opposite directions at the two ends indicate heavy tails (higher kurtosis) or light tails (lower kurtosis) relative to a normal distribution.
    -   **U-Shape / Inverted U-Shape (Skewness)**:
        -   **Right-skewed**: Points curve **above** the line at both ends (concave up).
        -   **Left-skewed**: Points curve **below** the line at both ends (concave down).
    -   **Outliers**: Isolated points far from the line indicate potential outliers.

By inspecting the Q-Q plot, you can see that the residuals of the stork model deviate noticeably from normality.



#### The Shapiro-Wilk Test



Visual inspection of Q-Q plots is subjective. The [Shapiro-Wilk test](https://en.wikipedia.org/wiki/Shapiro%E2%80%93Wilk_test) provides a formal hypothesis test for normality of the residuals:



In [1]:
from scipy.stats import shapiro

stat, p = shapiro(results.resid)
print(f"Shapiro-Wilk Test: W = {stat:.3f}, p = {p:.3f}")

if p > 0.05:
    print("Fail to reject normality (residuals are consistent with a normal distribution)")
else:
    print("Reject normality (residuals deviate significantly from a normal distribution)")

Here&rsquo;s how to interpret the results of the test:

1.  **Null Hypothesis (H0)**: The residuals are drawn from a normal distribution.
2.  **Alternative Hypothesis (H1)**: The residuals are not drawn from a normal distribution.
3.  **Test Statistic (W)**: Measures how closely the ordered sample values match expected normal order statistics. Values close to 1 indicate good agreement with normality.
4.  **p-value**:
    -   If $p > \alpha$ (commonly 0.05), we **fail to reject** $H_0$—there is insufficient evidence against normality (note: this does not &ldquo;prove&rdquo; normality).
    -   If $p \le \alpha$, we **reject** $H_0$—the residuals deviate significantly from a normal distribution.
5.  **Sample Size Considerations**:
    -   The Shapiro-Wilk test has relatively high power compared to other normality tests at small sample sizes, and like all hypothesis tests its power increases with sample size. With large sample sets, even minor, practically negligible deviations from normality can yield $p < 0.05$, so always pair the test with a Q-Q plot.
6.  **Practical Implications**: If the residuals are strongly non-normal (especially with small sample sizes), consider transforming the data (e.g., log-transform), using robust regression, or using bootstrap confidence intervals.



### Adding the regression line, confidence interval, and prediction interval



Although the diagnostics above suggest caution when applying a simple linear model to the stork dataset, we continue to use it here for illustration. Beyond r<sup>2</sup> and the p-value, adding uncertainty bands to our regression plot gives a much clearer picture of the model&rsquo;s reliability:

-   The fitted regression line
-   The **confidence interval** (CI) for the mean response (uncertainty in where the true regression line lies)
-   The **prediction interval** (PI) for individual new observations (expected range for a new data point at a given `x`)



#### Accessing the interval data



Uncertainty bands visually illustrate the reliability of a linear model. There are two distinct intervals:

-   **Confidence Interval (CI)**: quantifies uncertainty in the **mean** value of `y` at a given `x`.
-   **Prediction Interval (PI)**: quantifies the expected range for a **single new observation** of `y` at a given `x`. It is always wider than the confidence interval because it accounts for both uncertainty in the regression line and irreducible scatter of individual observations around that line.

Both intervals depend on the chosen significance level &alpha; , where $1 - \alpha$ is the confidence level (for example, $\alpha = 0.05$ corresponds to a 95% confidence/prediction interval).

We can retrieve both intervals from our fitted `statsmodels` results object in one line:



In [1]:
pred = results.get_prediction(df).summary_frame(alpha=0.05)

# pred["mean"]          -> fitted values
# pred["mean_ci_lower"] -> lower bound of 95% confidence interval (mean)
# pred["mean_ci_upper"] -> upper bound of 95% confidence interval (mean)
# pred["obs_ci_lower"]  -> lower bound of 95% prediction interval (new obs)
# pred["obs_ci_upper"]  -> upper bound of 95% prediction interval (new obs)

#### Plotting the interval bands



Matplotlib&rsquo;s `fill_between()` method shades the region between an upper and lower boundary:



In [1]:
ax.fill_between(df["X"], pred["mean_ci_lower"], pred["mean_ci_upper"], alpha=0.25, color="C1")

Note the `alpha` keyword here has nothing to do with the statistical significance level &alpha;; in Matplotlib, `alpha` controls opacity (1 = fully opaque, 0 = fully transparent). Also make sure `df` is sorted by `X` before calling `fill_between` so the shaded polygon is drawn cleanly from left to right.



### Creating the Stork Figure



Now let&rsquo;s put it all together in a concise, streamlined script. Note that we set the Matplotlib style **before** creating the figure, sort `df` by `X` and place the summary text using axis coordinates (`transform=ax.transAxes`) so it works regardless of the data range, i.e., rather than specifying your text locaction in data coordinates, you specify it as a percentage of the axis length.



In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.formula.api as smf

# Load, rename, drop missing values, and sort by X for clean band plotting
df = (
    pd.read_csv("storks_vs_birth_rate.csv")
    .set_axis(["Y", "X"], axis=1)
    .dropna()
    .sort_values("X")
)

# Fit OLS model and compute 95% confidence & prediction intervals
results = smf.ols("Y ~ X", data=df).fit()
pred = results.get_prediction(df).summary_frame(alpha=0.05)

# Plot
plt.style.use("ggplot")
fig, ax = plt.subplots(dpi=120, layout="constrained")

# add confidence bands
ax.fill_between(df["X"], pred["obs_ci_lower"], pred["obs_ci_upper"], alpha=0.12, color="C1", label="95% Prediction Interval")
ax.fill_between(df["X"], pred["mean_ci_lower"], pred["mean_ci_upper"], alpha=0.25, color="C1", label="95% Confidence Interval")
# plot regression line
ax.plot(df["X"], pred["mean"], color="C1", label="OLS Fit")
# plot data
ax.scatter(df["X"], df["Y"], color="C0", label="Data")

# add regression paramters
summary_text = (
    f"y = {results.params['Intercept']:.2f} + {results.params['X']:.4f}x\n"
    f"$R^2$ = {results.rsquared:.2f}\n"
    f"p = {results.pvalues['X']:.4f}"
)
ax.text(0.05, 0.95, summary_text, transform=ax.transAxes, va="top")
ax.set(xlabel="Stork Pairs", ylabel=r"Newborn Babies [$10^3$]")
ax.legend(loc="lower right")
fig.savefig("stork_new.png")
plt.show()

### References



-   Robert Matthews, 2000. Storks Devilver Babies (p = 0.008), Teaching Statistics, vol22:2, p. 36-38, [https://doi.org/10.1111/1467-9639.00013](https://doi.org/10.1111/1467-9639.00013),

